# Tamargi.ai — Core Hybrid RAG

This notebook contains only the **core Retrieval-Augmented Generation pipeline**.

It intentionally does **not** include:
- Egyptian Arabic processing
- patient profiles
- symptom analysis
- disease diagnosis
- treatment recommendation
- memory
- wearable data
- agent tools

The goal of this version is to build and understand a strong medical-document RAG first.

Pipeline:

`PDFs → Cleaning → Chunking → Embeddings + BM25 → RRF → Reranker → Gemini → Grounded Answer + Sources`


## 1. Install Packages
Install the libraries used for PDF reading, embeddings, sparse retrieval, reranking, evaluation, and Gemini.

In [ ]:
import sys

!"{sys.executable}" -m pip install -U pypdf pandas numpy sentence-transformers rank-bm25 scikit-learn google-genai

## 2. Imports
Import the libraries needed by the notebook.

In [1]:
from pypdf import PdfReader
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from google import genai
from google.genai import types
from getpass import getpass

import pandas as pd
import numpy as np
import torch
import re
import os
import time

## 3. Paths
Define the raw PDF folder and the processed-data folder.

In [2]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw:", RAW_DIR.resolve())
print("Processed:", PROCESSED_DIR.resolve())

Raw: C:\Users\user\Downloads\tamargi-pharma-rag\data\raw
Processed: C:\Users\user\Downloads\tamargi-pharma-rag\data\processed


## 4. Load PDFs
Find every PDF inside the raw-data folder.

In [3]:
pdf_files = list(RAW_DIR.rglob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print(pdf.name)

Number of PDFs: 13
egypt_antimicrobial_formulary_2023.pdf
egypt_antiretroviral_formulary_2026.pdf
egypt_blood_disorders_formulary_2025.pdf
egypt_cardiovascular_formulary_2024.pdf
egypt_conventional_anticancer_formulary_2024.pdf
egypt_endocrine_formulary_2024.pdf
egypt_gastrointestinal_formulary_2025.pdf
egypt_nervous_system_formulary_2025.pdf
egypt_respiratory_formulary_2026.pdf
egypt_targeted_anticancer_formulary_2025.pdf
egypt_otc_drug_list_2026.pdf
egypt_do_not_crush_medications_2026.pdf
egypt_high_alert_medications_2025.pdf


## 5. Extract Text
Read every PDF page and keep the file name, page number, and extracted text.

In [4]:
def extract_pdf(pdf_path):

    reader = PdfReader(pdf_path)
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text is None:
            text = ""

        pages.append({
            "file": pdf_path.name,
            "page": page_number,
            "text": text
        })

    return pages

## 6. Clean Text
Remove repeated headers and PDF noise while preserving medically important numbers, units, drug names, and punctuation.

In [5]:
def clean_text(text):

    if text == "":
        return ""

    text = re.sub(r"^\s*\d+\s*", "", text)

    text = text.replace("Egyptian Drug Formulary", "")
    text = text.replace("Egyptian National Formulary-Antimicrobials", "")

    text = re.sub(
        r"Code:\s*EDA\.DUPP\.\s*Formulary\.\d+",
        "",
        text
    )

    text = re.sub(
        r"Version\s*1\.0\s*/\s*/?\s*\d{4}",
        "",
        text
    )

    text = text.replace("\u00ad", "")
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")
    text = text.replace("\\.", ".")

    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("’", "'")

    text = re.sub(
        r"([A-Za-z]+)-\n([a-z]+)",
        r"\1\2",
        text
    )

    text = re.sub(r"\.{4,}", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## 7. Build Documents DataFrame
Extract and clean every page, then place the result in a pandas DataFrame.

In [6]:
all_pages = []

for pdf in pdf_files:

    print("Processing:", pdf.name)

    pages = extract_pdf(pdf)

    for page in pages:

        page["text"] = clean_text(page["text"])
        all_pages.append(page)

documents_df = pd.DataFrame(all_pages)

documents_df = documents_df[
    documents_df["text"].str.strip() != ""
].reset_index(drop=True)

print("Total pages:", len(documents_df))

documents_df.head()

Processing: egypt_antimicrobial_formulary_2023.pdf
Processing: egypt_antiretroviral_formulary_2026.pdf
Processing: egypt_blood_disorders_formulary_2025.pdf
Processing: egypt_cardiovascular_formulary_2024.pdf
Processing: egypt_conventional_anticancer_formulary_2024.pdf
Processing: egypt_endocrine_formulary_2024.pdf
Processing: egypt_gastrointestinal_formulary_2025.pdf
Processing: egypt_nervous_system_formulary_2025.pdf
Processing: egypt_respiratory_formulary_2026.pdf
Processing: egypt_targeted_anticancer_formulary_2025.pdf
Processing: egypt_otc_drug_list_2026.pdf
Processing: egypt_do_not_crush_medications_2026.pdf
Processing: egypt_high_alert_medications_2025.pdf
Total pages: 2084


,file,page,text
0,egypt_antimicrobial_formulary_2023.pdf,1,I\n\nCentral Administration of Pharmaceutical ...
1,egypt_antimicrobial_formulary_2023.pdf,2,II\n\nTable of Contents\nPage Content\nIII Pre...
2,egypt_antimicrobial_formulary_2023.pdf,3,III\n\nPreface\nThe Egyptian Antimicrobial Dru...
3,egypt_antimicrobial_formulary_2023.pdf,4,IV\n\nEgyptian Antimicrobial drug formulary ma...
4,egypt_antimicrobial_formulary_2023.pdf,5,V\n\n• Reserve: This group includes antibiotic...


## 8. Chunking
Split long pages into smaller overlapping text chunks. Overlap prevents information near a chunk boundary from being lost.

In [7]:
def create_chunks(text, chunk_size=120, overlap=20):

    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk_words = words[start:end]
        chunk = " ".join(chunk_words)

        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [8]:
def build_chunks_df(documents_df, chunk_size=120, overlap=20, min_words=25):

    all_chunks = []

    for index, row in documents_df.iterrows():

        chunks = create_chunks(
            row["text"],
            chunk_size=chunk_size,
            overlap=overlap
        )

        for chunk_number, chunk in enumerate(chunks, start=1):

            word_count = len(chunk.split())

            if word_count >= min_words:

                all_chunks.append({
                    "file": row["file"],
                    "page": row["page"],
                    "chunk_id": chunk_number,
                    "text": chunk,
                    "word_count": word_count
                })

    return pd.DataFrame(all_chunks)

chunks_df = build_chunks_df(documents_df)

print("Chunks:", len(chunks_df))

chunks_df.head()

Chunks: 6310


,file,page,chunk_id,text,word_count
0,egypt_antimicrobial_formulary_2023.pdf,1,1,I Central Administration of Pharmaceutical Car...,26
1,egypt_antimicrobial_formulary_2023.pdf,2,1,II Table of Contents Page Content III Preface ...,41
2,egypt_antimicrobial_formulary_2023.pdf,3,1,III Preface The Egyptian Antimicrobial Drug Fo...,120
3,egypt_antimicrobial_formulary_2023.pdf,4,1,IV Egyptian Antimicrobial drug formulary manua...,120
4,egypt_antimicrobial_formulary_2023.pdf,4,2,labeled indications 6. For antibiotics: includ...,92


## 9. GPU
Use CUDA automatically when an NVIDIA GPU is available.

In [9]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

CUDA available: False
Device: cpu


## 10. Embedding Model
The multilingual E5 model converts each chunk into a dense semantic vector.

In [10]:
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

print("Embedding model:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model: intfloat/multilingual-e5-base


## 11. Build Embeddings
E5 expects `passage:` before documents and `query:` before search queries.

In [11]:
passages = []

for text in chunks_df["text"]:
    passages.append("passage: " + text)

embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/395 [00:00<?, ?it/s]

Embedding shape: (6310, 768)


## 12. Dense Retrieval
Dense search finds chunks with similar meaning, even when the exact same words are not used.

In [12]:
def dense_search(query, top_k=20):

    query_embedding = model.encode(
        ["query: " + query],
        normalize_embeddings=True
    )[0]

    scores = embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):

        results.append({
            "index": int(index),
            "rank": rank,
            "dense_score": float(scores[index])
        })

    return results

## 13. BM25 Sparse Retrieval
BM25 is useful for exact drug names, medical terms, abbreviations, and keywords.

In [13]:
def tokenize(text):

    text = text.lower()

    return re.findall(
        r"[a-z0-9]+",
        text
    )

tokenized_chunks = []

for text in chunks_df["text"]:
    tokenized_chunks.append(tokenize(text))

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 ready")

BM25 ready


In [14]:
def bm25_search(query, top_k=20):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):

        results.append({
            "index": int(index),
            "rank": rank,
            "bm25_score": float(scores[index])
        })

    return results

## 14. Hybrid Retrieval with RRF
Reciprocal Rank Fusion combines dense semantic retrieval and BM25 without requiring their raw scores to be on the same scale.

In [15]:
def hybrid_search(
    query,
    dense_k=20,
    bm25_k=20,
    final_k=20,
    rrf_k=60,
    dense_weight=1.0,
    bm25_weight=1.0
):

    dense_results = dense_search(
        query,
        top_k=dense_k
    )

    bm25_results = bm25_search(
        query,
        top_k=bm25_k
    )

    rrf_scores = {}

    for result in dense_results:

        index = result["index"]
        rank = result["rank"]

        rrf_scores[index] = (
            rrf_scores.get(index, 0)
            + dense_weight / (rrf_k + rank)
        )

    for result in bm25_results:

        index = result["index"]
        rank = result["rank"]

        rrf_scores[index] = (
            rrf_scores.get(index, 0)
            + bm25_weight / (rrf_k + rank)
        )

    sorted_results = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for rank, item in enumerate(sorted_results[:final_k], start=1):

        index = item[0]
        score = item[1]

        row = chunks_df.iloc[index]

        results.append({
            "index": index,
            "rank": rank,
            "rrf_score": score,
            "file": row["file"],
            "page": int(row["page"]),
            "chunk_id": int(row["chunk_id"]),
            "text": row["text"]
        })

    return results

## 15. Reranker
The cross-encoder reads the query and each candidate chunk together and gives a stronger relevance score.

In [16]:
RERANKER_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker = CrossEncoder(
    RERANKER_MODEL,
    device=device
)

print("Reranker:", RERANKER_MODEL)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


README.md: 0.00B [00:00, ?B/s]

Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [17]:
def rerank_results(query, results, top_k=5):

    pairs = []

    for result in results:
        pairs.append([
            query,
            result["text"]
        ])

    scores = reranker.predict(pairs)

    reranked = []

    for result, score in zip(results, scores):

        new_result = result.copy()
        new_result["reranker_score"] = float(score)

        reranked.append(new_result)

    reranked = sorted(
        reranked,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked[:top_k]

## 16. Final Retriever
The final retriever runs Dense + BM25 + RRF and then reranks the best candidates.

In [18]:
DENSE_K = 20
BM25_K = 20
RRF_K = 60
CANDIDATE_K = 20
FINAL_K = 5

def retrieve(query):

    hybrid_results = hybrid_search(
        query,
        dense_k=DENSE_K,
        bm25_k=BM25_K,
        final_k=CANDIDATE_K,
        rrf_k=RRF_K
    )

    final_results = rerank_results(
        query,
        hybrid_results,
        top_k=FINAL_K
    )

    return final_results

## 17. Test Retrieval
Always inspect retrieval before connecting the LLM.

In [19]:
query = "What are the contraindications of Anidulafungin?"

results = retrieve(query)

for rank, result in enumerate(results, start=1):

    print("Rank:", rank)
    print("File:", result["file"])
    print("Page:", result["page"])
    print("RRF Score:", result["rrf_score"])
    print("Reranker Score:", result["reranker_score"])
    print(result["text"])
    print("-" * 100)

Rank: 1
File: egypt_antimicrobial_formulary_2023.pdf
Page: 52
RRF Score: 0.031009615384615385
Reranker Score: 2.9230902194976807
200 mg on day 1; subsequent dosing: 100 mg once daily; treatment should continue until 14 days after last positive culture. Candidiasis, esophageal (alternative agent): IV: 200 mg daily; may transition to oral fluconazole therapy once oral intake tolerable. Transition to an oral antifungal once patient tolerates oral intake if susceptibility allows; total antifungal duration is 14 to 28 days Dosage adjustment Dosing: Renal Impairment: Adult No dosage adjustment necessary, including dialysis patients. Dosing: Hepatic Impairment: Adult No dosage adjustment necessary. Contraindications Hypersensitivity to anidulafungin, other echinocandins, or any component of the formulation; known or suspected hereditary fructose intolerance. Adverse Drug Reactions >10%: Cardiovascular: Hypotension (15%), hypertension (12%), peripheral edema (11%) Central nervous system: Insom

## 18. Retrieval Evaluation

For a serious final evaluation, manually label the relevant pages or chunks for each question.

This starter evaluation uses known relevant file/page labels so Precision@K, Recall@K, and MRR have a clear meaning.


In [20]:
evaluation_queries = [
    {
        "query": "What are the contraindications of Anidulafungin?",
        "relevant_pages": [
            ("egypt_antimicrobial_formulary_2023.pdf", 52)
        ]
    }
]

print("Evaluation questions:", len(evaluation_queries))

Evaluation questions: 1


In [21]:
def evaluate_query(item, k=5):

    results = retrieve(item["query"])[:k]

    relevant_set = set(item["relevant_pages"])

    retrieved_set = set()

    first_relevant_rank = None

    for rank, result in enumerate(results, start=1):

        pair = (
            result["file"],
            result["page"]
        )

        retrieved_set.add(pair)

        if pair in relevant_set and first_relevant_rank is None:
            first_relevant_rank = rank

    true_positive = len(
        retrieved_set.intersection(relevant_set)
    )

    precision = true_positive / k

    recall = true_positive / len(relevant_set)

    if first_relevant_rank is None:
        reciprocal_rank = 0
    else:
        reciprocal_rank = 1 / first_relevant_rank

    return precision, recall, reciprocal_rank


precision_scores = []
recall_scores = []
mrr_scores = []

for item in evaluation_queries:

    precision, recall, reciprocal_rank = evaluate_query(item)

    precision_scores.append(precision)
    recall_scores.append(recall)
    mrr_scores.append(reciprocal_rank)

print("Precision@5:", np.mean(precision_scores))
print("Recall@5:", np.mean(recall_scores))
print("MRR:", np.mean(mrr_scores))

Precision@5: 0.2
Recall@5: 1.0
MRR: 1.0


## 19. Gemini API
Gemini is only the answer generator. Retrieval remains the source of medical evidence.

In [22]:
if "GEMINI_API_KEY" not in os.environ:

    os.environ["GEMINI_API_KEY"] = getpass(
        "Gemini API Key: "
    )

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

GEMINI_MODEL = "gemini-3.5-flash-lite"

print("Gemini ready")

Gemini API Key:  ········


Gemini ready


## 20. Strict Grounding Prompt
The model is not allowed to answer using unsupported medical knowledge.

In [23]:
STRICT_GROUNDING_PROMPT = '''
You are Tamargi.ai, a medication evidence assistant.

Answer only from the retrieved evidence included in the current request.

Do not use outside medical knowledge.
Do not guess.
Do not invent facts or citations.

Do not provide a dosage, contraindication, interaction, warning,
pregnancy recommendation, treatment recommendation, or other medical
claim unless it is explicitly supported by the retrieved evidence.

Use citations such as [E1], [E2], [E3] after factual claims.

If the evidence is insufficient, answer exactly:

"The retrieved evidence is insufficient to answer this question safely."
'''.strip()

## 21. Build Evidence
Each retrieved chunk receives an evidence ID and keeps its original file and page.

In [24]:
def build_evidence(results):

    blocks = []

    for i, result in enumerate(results, start=1):

        block = (
            "[E" + str(i) + "]\n"
            + "Source: " + result["file"] + "\n"
            + "Page: " + str(result["page"]) + "\n"
            + "Text: " + result["text"]
        )

        blocks.append(block)

    return "\n\n".join(blocks)

## 22. Grounded Generation
Send only the user question and retrieved evidence to Gemini.

In [25]:
def generate_grounded_answer(query, results):

    evidence = build_evidence(results)

    prompt = (
        "Question:\n"
        + query
        + "\n\nRetrieved evidence:\n"
        + evidence
    )

    for attempt in range(3):

        try:

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=STRICT_GROUNDING_PROMPT
                )
            )

            return response.text

        except Exception as e:

            if attempt == 2:
                raise e

            print("Gemini unavailable. Retrying...")
            time.sleep(5)

## 23. Verified Sources
Sources are created by Python from the retrieval results instead of being invented by the LLM.

In [26]:
def build_sources(results):

    sources = []

    for i, result in enumerate(results, start=1):

        source = (
            "[E"
            + str(i)
            + "] "
            + result["file"]
            + " | Page "
            + str(result["page"])
        )

        sources.append(source)

    return "\n".join(sources)

## 24. Final RAG Pipeline
This is the function that connects retrieval and grounded generation.

In [27]:
def tamargi_rag(query):

    results = retrieve(query)

    answer = generate_grounded_answer(
        query,
        results
    )

    return {
        "query": query,
        "answer": answer,
        "results": results,
        "sources": build_sources(results)
    }

## 25. Final Test

In [28]:
result = tamargi_rag(
    "What are the contraindications of Anidulafungin?"
)

print(result["answer"])
print()
print("Verified Sources")
print(result["sources"])

Based on the retrieved evidence, the contraindications for anidulafungin are:

* Hypersensitivity to anidulafungin, other echinocandins, or any component of the formulation [E1].
* Known or suspected hereditary fructose intolerance [E1].

Verified Sources
[E1] egypt_antimicrobial_formulary_2023.pdf | Page 52
[E2] egypt_antimicrobial_formulary_2023.pdf | Page 52
[E3] egypt_antimicrobial_formulary_2023.pdf | Page 337
[E4] egypt_antimicrobial_formulary_2023.pdf | Page 53
[E5] egypt_endocrine_formulary_2024.pdf | Page 80


## 26. Grounding Safety Test
Ask a question that the retrieved evidence should not support. A safe system should refuse instead of hallucinating.

In [29]:
result = tamargi_rag(
    "Does Anidulafungin cure hypertension?"
)

print(result["answer"])
print()
print("Verified Sources")
print(result["sources"])

The retrieved evidence is insufficient to answer this question safely.

Verified Sources
[E1] egypt_antimicrobial_formulary_2023.pdf | Page 52
[E2] egypt_nervous_system_formulary_2025.pdf | Page 98
[E3] egypt_antimicrobial_formulary_2023.pdf | Page 53
[E4] egypt_antimicrobial_formulary_2023.pdf | Page 53
[E5] egypt_endocrine_formulary_2024.pdf | Page 116


In [30]:
evaluation_queries = [
    {
        "query": "What are the contraindications of Anidulafungin?",
        "relevant_pages": [
            ("egypt_antimicrobial_formulary_2023.pdf", 52)
        ]
    },
    {
        "query": "What are the drug interactions of Anidulafungin?",
        "relevant_pages": [
            ("egypt_antimicrobial_formulary_2023.pdf", 53)
        ]
    },
    {
        "query": "What is the dosage of Anidulafungin?",
        "relevant_pages": [
            ("egypt_antimicrobial_formulary_2023.pdf", 52)
        ]
    }
]

In [31]:
def evaluate_query(item, k=5):

    results = retrieve(item["query"])[:k]

    relevant_set = set(item["relevant_pages"])

    retrieved_set = set()

    first_relevant_rank = None

    for rank, result in enumerate(results, start=1):

        pair = (
            result["file"],
            result["page"]
        )

        retrieved_set.add(pair)

        if pair in relevant_set and first_relevant_rank is None:
            first_relevant_rank = rank

    true_positive = len(
        retrieved_set.intersection(relevant_set)
    )

    precision = true_positive / k

    recall = true_positive / len(relevant_set)

    if first_relevant_rank is None:
        reciprocal_rank = 0
    else:
        reciprocal_rank = 1 / first_relevant_rank

    return precision, recall, reciprocal_rank

In [32]:
precision_scores = []
recall_scores = []
mrr_scores = []

for item in evaluation_queries:

    precision, recall, mrr = evaluate_query(item)

    precision_scores.append(precision)
    recall_scores.append(recall)
    mrr_scores.append(mrr)

print("Precision@5:", np.mean(precision_scores))
print("Recall@5:", np.mean(recall_scores))
print("MRR:", np.mean(mrr_scores))

Precision@5: 0.20000000000000004
Recall@5: 1.0
MRR: 1.0


# What You Should Be Able to Explain

After finishing this notebook, you should understand:

1. Why PDF text is cleaned.
2. Why documents are split into chunks.
3. What embeddings are.
4. The difference between dense search and BM25.
5. Why hybrid retrieval uses both.
6. What Reciprocal Rank Fusion does.
7. Why a reranker is added after retrieval.
8. What Precision@K, Recall@K, and MRR measure.
9. Why Gemini is not used as the medical knowledge source.
10. How strict grounding reduces hallucination.
11. Why file/page citations are kept with every retrieved chunk.

Advanced patient, disease, Arabic, safety-engine, memory, and agent features can be added later without changing this core RAG idea.
